# **Training Pipeline trên TPU v5e (Google Colab)**

Quy trình huấn luyện seq2seq cho tóm tắt văn bản tiếng Việt trên **TPU v5e-1**.

### Lưu ý quan trọng khi dùng TPU:
- Dùng **bf16** (TPU v5e hỗ trợ native bfloat16)
- **Không dùng** CUDA quantization (BitsAndBytes không hỗ trợ TPU)
- Cần cài **torch_xla** để PyTorch giao tiếp với TPU
- HuggingFace Trainer tự động detect TPU khi có torch_xla

---
## 1. Kiểm tra TPU & Cài đặt

In [ ]:
# Kiểm tra TPU có sẵn không
import os

# Kiểm tra biến môi trường TPU
tpu_name = os.environ.get('TPU_NAME', os.environ.get('COLAB_TPU_ADDR', None))
print(f"TPU_NAME: {tpu_name}")

# Kiểm tra torch_xla
try:
    import torch_xla
    import torch_xla.core.xla_model as xm

    device = xm.xla_device()
    print(f"✅ TPU sẵn sàng! Device: {device}")
    print(f"   torch_xla version: {torch_xla.__version__}")
except Exception as e:
    print(f"❌ Không tìm thấy TPU: {e}")
    print("Hãy chắc chắn đã chọn Runtime > Change runtime type > TPU v5e")

In [ ]:
import os

REPO_URL = "https://github.com/dungcony/sumarization.git"

# Khắc phục lỗi clone lồng nhau (lỗi 2 thư mục sumarization)
if os.path.exists(".git") and "sumarization" in os.getcwd():
    print("Đang cập nhật code mới nhất...")
    !git pull
else:
    print("Đang tải mã nguồn...")
    !git clone {REPO_URL} sumarization
    %cd sumarization

In [ ]:
# Cài đặt dependencies
# Colab đã có sẵn torch, torch_xla, transformers, datasets
# Chỉ cần cài thêm những gì thiếu
%pip install -e . 2>&1 | tail -5

---
## 2. Import

In [ ]:
import warnings
warnings.filterwarnings('ignore') # Ẩn các cảnh báo phiền phức

from __future__ import annotations

import time
from pathlib import Path

from transformers import (
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

from src.config import load_config, apply_overrides, config_to_dict
from src.data import load_and_preprocess
from src.evaluator import build_compute_metrics
from src.model import (
    apply_lora,
    enable_gradient_checkpointing,
    freeze_encoder,
    load_model,
    load_tokenizer,
)
from src.utils import (
    format_duration,
    format_number,
    save_json,
    set_seed,
    setup_logger,
    count_parameters,
)

logger = setup_logger("notebook")
print("✅ Import thành công!")

---
## 3. Cấu hình phần cứng  (đọc phần cứng để áp dụng gput hoặc tpu)

### Train lần 1 với các file tổng hợp

In [ ]:
# ============================================================
# TRAIN LẦN 1: Học nền tảng (file Parquet tổng hợp + GPU)
# ============================================================

# Chọn config cơ sở
CONFIG_FILE = "configs/vit5_base.yaml"

config = load_config(CONFIG_FILE)

# Tự động phát hiện đang dùng TPU hay GPU
is_tpu = "TPU_NAME" in os.environ or "COLAB_TPU_ADDR" in os.environ

# Ghi đè các thiết lập cho Lần 1
config = apply_overrides(config, {
    # Dữ liệu: Dùng file Parquet tổng hợp (đã có sẵn trong 3 thư mục con)
    "data.train_file": "data/fine-turn/train/*.parquet",
    "data.valid_file": "data/fine-turn/validation/*.parquet",
    "data.test_file": "data/fine-turn/test/*.parquet",

    # Precision & Optimizer
    "training.precision": "bf16" if is_tpu else "auto",
    "training.optim": "adafactor" if is_tpu else "adamw_torch",

    # Thư mục lưu kết quả Lần 1
    "training.output_dir": "/content/outputs/vit5_base_model" if not os.path.exists(
        "/kaggle") else "/kaggle/working/outputs/vit5_base_model",

    # Batch size
    "training.per_device_train_batch_size": 8 if is_tpu else 4,
    "training.per_device_eval_batch_size": 16 if is_tpu else 8,
    "training.gradient_accumulation_steps": 1 if is_tpu else 2,


    # --- CỤM CHẠY THỬ ---
    "data.max_train_samples": 200,   
    "data.max_eval_samples": 50,     
    "training.max_steps": 20,
    "training.logging_steps": 2,     # <--- Thêm ở đây
    "training.eval_steps": 5,        # <--- Thêm ở đây

})

print(f"🔹 TRAIN LẦN 1: Học nền tảng (Parquet tổng hợp)")
print(f"Hardware:   {'TPU' if is_tpu else 'GPU'}")
print(f"Model:      {config.model.name_or_path}")
print(f"Train data: {config.data.train_file}")
print(f"Precision:  {config.training.precision}")
print(f"Batch size: {config.training.per_device_train_batch_size}")
print(f"LR:         {config.training.learning_rate}")
print(f"Optimizer:  {config.training.optim}")
print(f"Output:     {config.training.output_dir}")


### train lần 2 với các file theo chủ đề y tế (.csv)

In [ ]:
# ============================================================
# TRAIN LẦN 2: Học chuyên ngành Y tế (file CSV + TPU)
# ============================================================

# Chọn config cơ sở
CONFIG_FILE = "configs/vit5_base.yaml"

config = load_config(CONFIG_FILE)

# Tự động phát hiện đang dùng TPU hay GPU
is_tpu = "TPU_NAME" in os.environ or "COLAB_TPU_ADDR" in os.environ

# Xác định đường dẫn model Lần 1 (tự nhận diện Colab hay Kaggle)
PHASE1_MODEL = "/content/outputs/vit5_base_model/best" if not os.path.exists(
    "/kaggle") else "/kaggle/working/outputs/vit5_base_model/best"

# Ghi đè các thiết lập cho Lần 2
config = apply_overrides(config, {
    # Nạp lại bộ não đã thông minh từ Lần 1
    "model.name_or_path": PHASE1_MODEL,

    # Dữ liệu: Dùng file CSV chuyên ngành Y tế
    "data.train_file": "data/fine-turn/train/*.csv",
    "data.valid_file": "data/fine-turn/validation/*.csv",
    "data.test_file": "data/fine-turn/test/*.csv",

    # Precision & Optimizer
    "training.precision": "bf16" if is_tpu else "auto",
    "training.optim": "adafactor" if is_tpu else "adamw_torch",

    # Thư mục lưu kết quả Lần 2 (KHÁC với Lần 1 để không ghi đè)
    "training.output_dir": "/content/outputs/vit5_medical_model" if not os.path.exists(
        "/kaggle") else "/kaggle/working/outputs/vit5_medical_model",

    # Batch size
    "training.per_device_train_batch_size": 8 if is_tpu else 4,
    "training.per_device_eval_batch_size": 16 if is_tpu else 8,
    "training.gradient_accumulation_steps": 1 if is_tpu else 2,

    # Giảm tốc độ học (1e-5) để ngấm từ vựng y khoa mà không quên kiến thức cũ
    "training.learning_rate": 1e-5,

     # --- CỤM CHẠY THỬ ---
    "data.max_train_samples": 200,   
    "data.max_eval_samples": 50,     
    "training.max_steps": 20,
    "training.logging_steps": 2,     # <--- Thêm ở đây
    "training.eval_steps": 5,        # <--- Thêm ở đây
})

print(f"🔸 TRAIN LẦN 2: Học chuyên ngành Y tế (CSV)")
print(f"Hardware:   {'TPU' if is_tpu else 'GPU'}")
print(f"Model:      {config.model.name_or_path}")
print(f"Train data: {config.data.train_file}")
print(f"Precision:  {config.training.precision}")
print(f"Batch size: {config.training.per_device_train_batch_size}")
print(f"LR:         {config.training.learning_rate}")
print(f"Optimizer:  {config.training.optim}")
print(f"Output:     {config.training.output_dir}")


---
## 4. Tải Model & Data

In [ ]:
# Thiết lập seed
set_seed(config.training.seed)

# Tải tokenizer
tokenizer = load_tokenizer(config.model)
print(f"✅ Tokenizer: vocab_size={tokenizer.vocab_size}")

In [ ]:
# Tải model
model = load_model(config.model, tokenizer, config.generation)

# Áp dụng gradient checkpointing (tiết kiệm bộ nhớ TPU)
if config.training.gradient_checkpointing:
    enable_gradient_checkpointing(model)

# Đóng băng encoder nếu cần
if config.training.freeze_encoder:
    freeze_encoder(model)

# Áp dụng LoRA nếu bật
model = apply_lora(model, config.lora)

params = count_parameters(model)
print(f"✅ Model loaded: {format_number(params['total'])} total, "
      f"{format_number(params['trainable'])} trainable ({params['trainable_percent']}%)")

In [ ]:
# Tải & tiền xử lý dữ liệu
datasets = load_and_preprocess(tokenizer, config.data)

print(f"✅ Train:      {len(datasets['train'])} samples")
print(f"✅ Validation: {len(datasets['validation'])} samples")
if 'test' in datasets:
    print(f"✅ Test:       {len(datasets['test'])} samples")


---
## 5. Cấu hình Trainer cho TPU

In [ ]:
tc = config.training
output_dir = Path(tc.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

# Xác định precision
fp16 = tc.precision == "fp16"
bf16 = tc.precision == "bf16"

training_args = Seq2SeqTrainingArguments(
    output_dir=tc.output_dir,
    seed=tc.seed,

    # Epochs & steps
    num_train_epochs=tc.num_train_epochs,
    max_steps=tc.max_steps,

    # Batch size
    per_device_train_batch_size=tc.per_device_train_batch_size,
    per_device_eval_batch_size=tc.per_device_eval_batch_size,
    gradient_accumulation_steps=tc.gradient_accumulation_steps,

    # Optimizer
    learning_rate=tc.learning_rate,
    weight_decay=tc.weight_decay,
    warmup_ratio=tc.warmup_ratio,
    lr_scheduler_type=tc.lr_scheduler_type,
    optim=tc.optim,  #thuật toán cập nhật kiến thức

    # Regularization
    label_smoothing_factor=tc.label_smoothing_factor,

    # Precision — TPU v5e dùng bf16
    fp16=fp16,
    bf16=bf16,

    # Evaluation & saving
    eval_strategy=tc.eval_strategy,  #số bước học cho mỗi lần kiểm tra
    eval_steps=tc.eval_steps,
    save_strategy=tc.save_strategy,  # số bước học cho mỗi lần lưu
    save_steps=tc.save_steps,
    save_total_limit=tc.save_total_limit,  # số bản lưu trữ được giữ lại
    logging_steps=tc.logging_steps,

    # Best model
    metric_for_best_model=tc.metric_for_best_model,
    greater_is_better=tc.greater_is_better,
    load_best_model_at_end=tc.load_best_model_at_end,  #chọn bản tốt nhất để lưu

    # Generation during eval
    predict_with_generate=True,  #yêu cầu tạo ra 1 bài báo và tóm tắt rồi mới chấm điểm
    generation_max_length=config.generation.max_length,  #số token tối đa được viết khi làm bài kiểm tra

    # Logging — dùng tensorboard
    report_to=["tensorboard"],
    logging_dir=str(output_dir / "logs"),

    # ===== TPU-specific =====
    # dataloader_drop_last giúp tránh batch cuối bị lẻ trên TPU
    dataloader_drop_last=True,
)

print(f"✅ TrainingArguments đã sẵn sàng")
print(f"   bf16={bf16}, fp16={fp16}")
print(f"   device: {training_args.device}")

In [ ]:
# Data collator (dynamic padding)
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
)

# Hàm tính ROUGE metrics
compute_metrics = build_compute_metrics(tokenizer)

# Callbacks
callbacks = []
if tc.early_stopping_patience > 0:
    callbacks.append(
        EarlyStoppingCallback(
            early_stopping_patience=tc.early_stopping_patience,
        )
    )
    print(f"✅ Early stopping: patience={tc.early_stopping_patience}")

# Xây dựng Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=datasets["train"],
    eval_dataset=datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=callbacks,
)

print("✅ Trainer đã sẵn sàng!")

---
## 6. Huấn luyện 🚀

In [ ]:
print("=" * 60)
print("BẮT ĐẦU HUẤN LUYỆN")
print("=" * 60)
print(f"  Model:       {config.model.name_or_path}")
print(f"  Epochs:      {tc.num_train_epochs}")
print(f"  Batch size:  {tc.per_device_train_batch_size}")
print(f"  Grad accum:  {tc.gradient_accumulation_steps}")
print(f"  LR:          {tc.learning_rate}")
print(f"  Optimizer:   {tc.optim}")
print(f"  Precision:   {tc.precision}")
print(f"  LoRA:        {config.lora.enabled}")
print("=" * 60)

start_time = time.time()

train_result = trainer.train(
    resume_from_checkpoint=tc.resume_from_checkpoint,
)

elapsed = time.time() - start_time
print(f"\n✅ Huấn luyện hoàn thành! Thời gian: {format_duration(elapsed)}")

---
## 7. Lưu model & Đánh giá

In [ ]:
# Lưu model tốt nhất
best_dir = output_dir / "best"
print(f"Đang lưu model tốt nhất tới: {best_dir}")
trainer.save_model(str(best_dir))
tokenizer.save_pretrained(str(best_dir))
print("✅ Đã lưu model!")

In [ ]:
# Đánh giá cuối cùng trên tập VALIDATION
print("Đang chạy đánh giá trên tập Validation...")
eval_results = trainer.evaluate(metric_key_prefix="eval")

print("\n" + "=" * 60)
print("KẾT QUẢ ĐÁNH GIÁ - TẬP VALIDATION")
print("=" * 60)
for key, value in sorted(eval_results.items()):
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
print("=" * 60)

# Đánh giá trên tập TEST (Bài thi Đại học - Độc lập hoàn toàn)
if 'test' in datasets:
    print("\nĐang đánh giá trên tập TEST...")
    test_results = trainer.evaluate(eval_dataset=datasets['test'], metric_key_prefix='test')
    print("\n" + "=" * 60)
    print("KẾT QUẢ ĐÁNH GIÁ - TẬP TEST (Khách quan nhất)")
    print("=" * 60)
    for k, v in sorted(test_results.items()):
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")
    print("=" * 60)
else:
    print("\n⚠️ Không tìm thấy tập TEST trong dữ liệu.")
    test_results = {}


In [ ]:
# Lưu metrics & config
train_metrics = train_result.metrics
train_metrics["train_runtime_formatted"] = format_duration(
    train_metrics.get("train_runtime", 0)
)

save_json(train_metrics, output_dir / "train_results.json")
save_json(eval_results, output_dir / "eval_results.json")
if test_results:
    save_json(test_results, output_dir / "test_results.json")
save_json(config_to_dict(config), output_dir / "resolved_config.json")

print("✅ Đã lưu tất cả metrics!")
print(f"   📁 {output_dir}")


---
## 8. (Tùy chọn) Copy model lên Google Drive (Chỉ dành cho Colab)
*Lưu ý: Nếu bạn chạy trên Kaggle thì KHÔNG chạy Cell này. Kaggle sẽ tự động đóng gói output để bạn tải về ở menu bên phải.*

In [ ]:
# Mount Google Drive & copy model để không bị mất khi disconnect
from google.colab import drive

drive.mount('/content/drive')

DRIVE_OUTPUT = "/content/drive/MyDrive/summarization_models/vit5_base_tpu"
!mkdir -p {DRIVE_OUTPUT}
!cp -r {output_dir / 'best'}/* {DRIVE_OUTPUT}/
!cp {output_dir}/*.json {DRIVE_OUTPUT}/

print(f"✅ Đã copy model lên Google Drive: {DRIVE_OUTPUT}")

---
## 9. (Tùy chọn) Test thử model

In [ ]:
from src.predict import summarize

test_text = """Thủ tướng Chính phủ vừa phê duyệt đề án phát triển ứng dụng 
trí tuệ nhân tạo tại Việt Nam giai đoạn 2025-2030. Theo đó, Việt Nam đặt 
mục tiêu trở thành một trong những trung tâm đổi mới sáng tạo về AI trong 
khu vực ASEAN. Đề án tập trung vào 5 lĩnh vực ưu tiên gồm y tế, giáo dục, 
nông nghiệp, giao thông và sản xuất công nghiệp."""

summary = summarize(
    text=test_text,
    model_path=str(best_dir),
    config=config,
)

print("📄 Bài gốc:")
print(test_text.strip())
print("\n📝 Tóm tắt:")
print(summary)